In [46]:
import os

In [47]:
from dotenv import load_dotenv

In [48]:
from pydantic import BaseModel, Field

In [49]:
from pydantic import BaseModel, Field

In [50]:
from langchain_groq import ChatGroq

In [51]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

In [52]:
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser

In [53]:
load_dotenv()

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [54]:
class ReviewData(BaseModel):
    complaint: str = Field(description="Main customer complaint")
    product_or_feature: str = Field(description="Product or feature mentioned")
    sentiment: str = Field(description="Customer sentiment")


parser = PydanticOutputParser(pydantic_object=ReviewData)

In [55]:
prompt1 = PromptTemplate(
    template="""
Analyze the customer review and extract the complaint,
product or feature, and sentiment.

Customer Review:
{review_text}

{format_instructions}
""",
    input_variables=["review_text"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    }
)

chain1 = prompt1 | llm | parser

In [56]:
prompt2 = ChatPromptTemplate.from_messages([
    ("system", "You are a support ticket writer."),
    ("human", """
Create a short support ticket summary for the internal team.

Complaint: {complaint}
Product or Feature: {product_or_feature}
Sentiment: {sentiment}
""")
])

chain2 = prompt2 | llm | StrOutputParser()

In [57]:
load_dotenv()

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

In [58]:
review_text = input("Enter customer review: ")

chain1 = prompt1 | llm | parser
chain2 = prompt2 | llm | StrOutputParser()

result1 = chain1.invoke({
    "review_text": review_text
})

print("\nStep 1 - Structured Complaint Data")
print(result1.model_dump())

result2 = chain2.invoke({
    "complaint": result1.complaint,
    "product_or_feature": result1.product_or_feature,
    "sentiment": result1.sentiment
})

print("\nStep 2 - Support Ticket Summary")
print(result2)


Step 1 - Structured Complaint Data
{'complaint': 'The payment button in my mobile app is not working and I cannot complete my order.', 'product_or_feature': 'Payment button in the mobile app', 'sentiment': 'Negative'}

Step 2 - Support Ticket Summary
**Support Ticket Summary**

- **Issue:** Payment button in the mobile app is not functioning; user cannot complete the order.  
- **Product/Feature:** Mobile app payment button.  
- **Sentiment:** Negative (user frustrated).  
- **Action Needed:** Investigate button functionality, check for recent updates or bugs, and provide a fix or workaround.
